# UX A/B Text Demo — Information Placement and Target Acquisition Analysis

This notebook analyzes **real experiment output only**. It treats the demo as a controlled comparison of **EARLY** versus **LATE** placement of answer-relevant information.

The scientific workflow separates three layers:

1. **Behavioral performance** — response accuracy and uncertainty.
2. **Target acquisition** — whether and when the predefined target phrase was fixated.
3. **Search effort** — fixation count, dwell, and scanpath distance before a response.

The notebook intentionally uses public `tobii-pytracker` analysis APIs wherever they match the scientific question. `DataLoader` provides session and gaze access, `FixationAnalyzer` extracts fixation events, and `ScanpathsAnalyzer` derives fixation-to-fixation transitions. The current public `BBoxAttentionAnalyzer` scores `image_bboxes`; therefore word-level target AOIs are scored locally from the `words` geometry stored in each trial rather than forcing an incompatible analyzer onto text data.

> **Interpretation rule:** this demo is suitable for descriptive and exploratory comparison. A confirmatory experiment requires participant-level replication, counterbalancing, and an inferential model that accounts for participant and item variability.

## Scientific context and experimental logic

This notebook implements the analysis strategy described in the demo documentation: [UX A/B Text Demo](../../../docs/basic_examples/ux_ab_text_demo.md). The experiment is a compact **information-placement study** in which answer-relevant content appears either **EARLY** or **LATE** in an otherwise short interface-style text.

### Experimental structure

- 12 response-gated text trials;
- 6 `EARLY` and 6 `LATE` trials;
- within each placement condition, 3 target answers are `YES` and 3 are `NO`;
- `I DON'T KNOW` is an allowed uncertainty response but is never the target class;
- the materials form matched EARLY/LATE pairs around practical interface questions;
- each stimulus is displayed as `QUESTION`, a blank separator line, and `TEXT`.

The scientific construct is **information access cost**. If decision-relevant information occurs later in the text, the reader typically has more material to traverse before reaching the critical phrase. Eye tracking makes this traversal directly observable instead of inferring it only from response time.

### Analysis model

The notebook separates the trial into three evidence layers:

1. **Behavioral outcome** — what response was selected and whether it was correct.
2. **Target acquisition** — whether the critical phrase was fixated and how quickly it was reached.
3. **Search and verification** — how much visual activity occurred before and after target acquisition.

| Documentation hypothesis | Operational measure in this notebook | Primary interpretation |
|---|---|---|
| H1 — Target acquisition | `target_seen`, target acquisition rate | Was answer-relevant information visually reached? |
| H2 — Placement effect | target TTFF: EARLY vs LATE | How quickly was relevant information accessed? |
| H3 — Search effort | fixations before target, scanpath distance | How much visual traversal preceded acquisition? |
| H4 — Behavior and gaze | accuracy by target acquisition | Is correct responding associated with visible access to the target? |

### Measurement caution

A fixation on the target phrase is evidence that gaze entered the target AOI; it is **not** direct evidence of comprehension. Likewise, longer dwell can reflect verification, ambiguity, rereading, or task difficulty. The notebook therefore interprets eye-movement metrics together with the response and the known experimental manipulation rather than treating any single measure as a latent construct such as cognitive load.

The demo documentation cites Rayner (1998), Goldberg & Kotval (1999), and Duchowski (2017) as methodological background for reading, HCI, and eye-tracking interpretation.


## 1. Analysis parameters

This cell is the **methodological control panel** for all event-dependent results. Fixation thresholds determine what counts as a stable fixation, so changing them can alter target acquisition, TTFF, dwell, and scanpath summaries. Keeping the values visible makes the analysis reproducible and supports later sensitivity checks.

For reports or publications, record these parameters exactly and avoid tuning them after inspecting condition differences. The optional sensitivity section later in the notebook is intended for robustness assessment, not post-hoc optimization.


In [ ]:
LANGUAGE = "en"  # "en" or "pl"
SELECTED_SESSION = None  # set to an explicit session directory name to reproduce a previous analysis
FIXATION_PARAMS = {
    "method": "dispersion",
    "dispersion_threshold": 50.0,
    "min_duration": 0.10,
}
RUN_FIXATION_SENSITIVITY = False
SENSITIVITY_DISPERSION_THRESHOLDS = [40.0, 50.0, 60.0]


## 2. Load the experiment output with `DataLoader`

The notebook analyzes collected sessions only; it never fabricates replacement data. `DataLoader` is used as the canonical interface to the experiment output so that trial metadata, gaze samples, screenshots, and saved geometry remain linked.

The default workflow selects the most recent valid session for convenience. For reproducible group work, explicitly select the session of interest and record its identifier in the analysis report.

> **tobii-pytracker support:** This step uses `CustomConfig` and `DataLoader` to open the experiment output and keep trial metadata, gaze samples, screenshots, and stored geometry aligned.


In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tobii_pytracker.configs.custom_config import CustomConfig
from tobii_pytracker.analyze import (
    DataLoader,
    FixationAnalyzer,
    ScanpathsAnalyzer,
)


def find_demo_root() -> Path:
    """Locate tobii-pytracker-demo from common Jupyter launch locations."""
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for candidate in candidates:
        if (candidate / "examples").is_dir() and (candidate / "output").is_dir():
            return candidate
        nested = candidate / "tobii-pytracker-demo"
        if (nested / "examples").is_dir() and (nested / "output").is_dir():
            return nested
    raise FileNotFoundError("Could not locate the tobii-pytracker-demo repository root.")


def newest_subject(loader: DataLoader) -> str:
    subjects = loader.get_subjects()
    if not subjects:
        raise FileNotFoundError(f"No experiment sessions found under {loader.output_root}")
    def mtime(subject: str) -> float:
        return (loader.output_root / subject / "data.csv").stat().st_mtime
    return max(subjects, key=mtime)


def prepare_session(config_path: Path, subject: str | None = None):
    """Load one real session with DataLoader; never synthesize replacement data."""
    config = CustomConfig(str(config_path))
    loader = DataLoader(config=config, root=DEMO_ROOT)
    selected = subject or newest_subject(loader)
    raw = loader.get_subject_data(selected, flatten=False).reset_index(drop=True)
    raw.insert(0, "set_name", selected)
    raw["slide_index"] = np.arange(len(raw), dtype=int)
    flat = loader.get_subject_data(selected, flatten=True)
    return loader, selected, raw, flat


def safe_parse(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    text = "" if value is None else str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default


def point_in_centered_bbox(x: float, y: float, bbox: dict, margin: float = 2.0) -> bool:
    try:
        cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
    except (KeyError, TypeError, ValueError):
        return False
    return (cx - w/2 - margin <= x <= cx + w/2 + margin and
            cy - h/2 - margin <= y <= cy + h/2 + margin)


def normalize_token(value) -> str:
    return re.sub(r"[^0-9a-ząćęłńóśźż]+", "", str(value).casefold())


def trial_gaze_counts(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    counts = pd.Series(0, index=range(n_trials), dtype=int)
    if not flat.empty and {"slide_index", "avg_gaze_x"}.issubset(flat.columns):
        observed = flat.dropna(subset=["avg_gaze_x", "avg_gaze_y"]).groupby("slide_index").size()
        for idx, count in observed.items():
            if int(idx) in counts.index:
                counts.loc[int(idx)] = int(count)
    return counts


def trial_observed_duration(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    duration = pd.Series(np.nan, index=range(n_trials), dtype=float)
    if not flat.empty and {"slide_index", "system_time"}.issubset(flat.columns):
        for idx, group in flat.dropna(subset=["system_time"]).groupby("slide_index"):
            if len(group) >= 2 and int(idx) in duration.index:
                duration.loc[int(idx)] = float(group["system_time"].max() - group["system_time"].min())
    return duration


def run_fixations(flat: pd.DataFrame, output_dir: Path, params: dict) -> pd.DataFrame:
    required = {"set_name", "slide_index", "avg_gaze_x", "avg_gaze_y", "system_time"}
    if flat.empty or not required.issubset(flat.columns):
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    clean = flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy()
    if clean.empty:
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    analyzer = FixationAnalyzer(output_dir, **params)
    return analyzer.analyze(clean)


def run_scanpaths(fixations: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    if fixations.empty:
        return pd.DataFrame(columns=["set_name","slide_index","distance"])
    return ScanpathsAnalyzer(output_dir).analyze(fixations, per="slide")


def summarize_fixations(fixations: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if fixations.empty:
        return base.assign(fixation_count=0, total_fixation_duration=0.0, mean_fixation_duration=np.nan)
    agg = (fixations.groupby("slide_index")
           .agg(fixation_count=("duration","size"),
                total_fixation_duration=("duration","sum"),
                mean_fixation_duration=("duration","mean"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"fixation_count":0,"total_fixation_duration":0.0})


def summarize_scanpaths(scanpaths: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if scanpaths.empty:
        return base.assign(scanpath_transition_count=0, scanpath_distance=0.0)
    agg = (scanpaths.groupby("slide_index")
           .agg(scanpath_transition_count=("distance","size"), scanpath_distance=("distance","sum"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"scanpath_transition_count":0,"scanpath_distance":0.0})

DEMO_ROOT = find_demo_root()
print(f"Demo repository: {DEMO_ROOT}")

In [ ]:
example_dir = DEMO_ROOT / "examples" / "ux_ab_demo"
config_candidates = sorted(example_dir.glob("config*.yaml"))
config_path = next(p for p in config_candidates if (p.stem.endswith("_pl")) == (LANGUAGE == "pl"))
dataset_name = "text_search.csv" if LANGUAGE == "en" else "text_search_pl.csv"
dataset_path = example_dir / "data" / dataset_name

loader, SESSION, raw, flat = prepare_session(config_path, SELECTED_SESSION)
stimuli = pd.read_csv(dataset_path)
lookup = stimuli.set_index("text", drop=False)

analysis_dir = loader.output_root / SESSION / "analysis_ux_ab_text_demo_v2"
analysis_dir.mkdir(exist_ok=True)

raw["stimulus_id"] = raw["input_data"].map(lookup["id"])
raw["pair_id"] = raw["stimulus_id"].astype(str).str.replace(r"^[eElL]", "", regex=True)
raw["condition"] = raw["input_data"].map(lookup["condition"])
raw["expected_answer"] = raw["input_data"].map(lookup["answer"])
raw["target_phrase"] = raw["input_data"].map(lookup["target_phrase"])
raw["response"] = raw["user_classification"].astype(str).str.casefold()
raw["correct"] = raw["response"] == raw["expected_answer"].astype(str).str.casefold()
raw["uncertain"] = raw["response"].isin(["none", "i don't know", "nie wiem"])

print(f"Session: {SESSION}")
print(f"Trials: {len(raw)} | Flattened gaze samples: {len(flat)}")
display(raw[["slide_index","condition","expected_answer","response","correct","target_phrase"]].head())

## 3. Reproducibility record and design integrity

Before any gaze statistic is interpreted, the notebook fingerprints the analyzed `data.csv` and stimulus table and records package versions. This creates a minimal provenance record that can be paired with exported results.

The design checks verify the expected 12-trial EARLY/LATE structure and balanced target responses. A failed design check should be investigated as a collection or configuration problem; it should not be silently repaired by dropping trials.


In [ ]:
import hashlib
import platform
from importlib.metadata import PackageNotFoundError, version as package_version


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(distribution: str) -> str:
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return "package-metadata-unavailable"


def provenance_table(session: str, data_csv: Path, dataset_path: Path, analysis_label: str) -> pd.DataFrame:
    record = {
        "analysis_label": analysis_label,
        "session": str(session),
        "data_csv_sha256": sha256_file(data_csv),
        "dataset_sha256": sha256_file(dataset_path),
        "python": platform.python_version(),
        "tobii_pytracker": installed_version("tobii-pytracker"),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }
    return pd.DataFrame([record])


def save_json(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

data_csv = loader.output_root / SESSION / "data.csv"
provenance = provenance_table(SESSION, data_csv, dataset_path, "ux_ab_text_demo_v2")
pair_design = (stimuli.assign(pair_id=stimuli["id"].astype(str).str.replace(r"^[eElL]", "", regex=True))
               .groupby("pair_id")["condition"].agg(lambda s: set(s.astype(str))))
complete_pairs = int(pair_design.apply(lambda values: values == {"EARLY", "LATE"}).sum())

design_checks = pd.DataFrame([
    {"check": "12 recorded trials", "passed": len(raw) == 12, "observed": len(raw), "expected": 12},
    {"check": "EARLY/LATE balance", "passed": raw["condition"].value_counts().to_dict() == {"EARLY": 6, "LATE": 6}, "observed": str(raw["condition"].value_counts().to_dict()), "expected": "EARLY=6, LATE=6"},
    {"check": "YES/NO balance in stimulus set", "passed": stimuli["answer"].str.casefold().value_counts().to_dict() == {"yes": 6, "no": 6}, "observed": str(stimuli["answer"].str.casefold().value_counts().to_dict()), "expected": "yes=6, no=6"},
    {"check": "6 complete matched EARLY/LATE stimulus pairs", "passed": complete_pairs == 6, "observed": complete_pairs, "expected": 6},
    {"check": "metadata mapping complete", "passed": raw[["condition","expected_answer","target_phrase","stimulus_id"]].notna().all().all(), "observed": int(raw[["condition","expected_answer","target_phrase","stimulus_id"]].isna().sum().sum()), "expected": 0},
])
display(provenance)
display(design_checks)
if not bool(design_checks["passed"].all()):
    warnings.warn("One or more design checks failed. Interpret the session as incomplete or non-canonical.")

## 4. Data-quality audit

Behavioral validity and eye-tracking availability are treated as separate dimensions. A trial can contain a valid response even when no usable gaze samples were recorded. Such a trial remains part of behavioral summaries but contributes no fixation- or AOI-based metric.

This distinction is essential for H4: a trial with missing gaze is **unknown target-acquisition status**, not evidence that the participant failed to inspect the target.


In [ ]:

gaze_counts = trial_gaze_counts(flat, len(raw))
observed_duration = trial_observed_duration(flat, len(raw))
quality = raw[["slide_index","stimulus_id","pair_id","condition","response","correct","uncertain"]].copy()
quality["response_present"] = quality["response"].notna() & ~quality["response"].astype(str).str.casefold().isin(["", "nan"])
quality["gaze_samples"] = quality["slide_index"].map(gaze_counts)
quality["usable_gaze"] = quality["gaze_samples"] > 0
quality["observed_gaze_duration_s"] = quality["slide_index"].map(observed_duration)
quality_overview = pd.DataFrame([
    {"metric": "recorded_trials", "value": len(quality)},
    {"metric": "trials_with_response", "value": int(quality["response"].notna().sum())},
    {"metric": "trials_with_usable_gaze", "value": int(quality["usable_gaze"].sum())},
    {"metric": "trials_without_usable_gaze", "value": int((~quality["usable_gaze"]).sum())},
    {"metric": "uncertain_responses", "value": int(quality["uncertain"].sum())},
])
display(quality_overview)
display(quality)
if (~quality["usable_gaze"]).any():
    warnings.warn("Eye-movement summaries exclude trials without usable gaze; behavioral rows remain in the dataset.")


## 5. Inspect one representative trial

Before aggregating conditions, inspect one real trial to understand the physical scale and structure of the collected data. `DataLoader.get_slide_data(..., flatten=True)` exposes sample-level gaze, and `plot_gaze()` overlays gaze on the recorded stimulus screenshot when available.

Use this step as a geometry sanity check: gaze should align with the visible text, and the question/text spacing should match the word-bbox representation used later for AOI scoring.

> **tobii-pytracker support:** `DataLoader.get_slide_data()` and `DataLoader.plot_gaze()` are used here for trial-level inspection before aggregation.


In [ ]:

if quality["usable_gaze"].any():
    representative = int(quality.loc[quality["usable_gaze"]].sort_values("gaze_samples", ascending=False).iloc[0]["slide_index"])
    slide = loader.get_slide_data(SESSION, representative, flatten=True)
    display(slide.head())
    loader.plot_gaze(SESSION, representative, gradient=True, show=True)
else:
    representative = None
    print("No trial contains usable gaze; screenshot-level gaze diagnostics are skipped.")


## 6. Fixations and scanpaths with built-in analyzers

`FixationAnalyzer` converts raw gaze into fixation events using the parameters defined above. `ScanpathsAnalyzer` then characterizes transitions between consecutive fixations. These built-in analyzers are preferred over local reimplementations so the notebook demonstrates the supported `tobii-pytracker` workflow.

For this experiment, fixation count and scanpath distance are interpreted as **descriptive search-effort measures**. They are not direct measurements of comprehension or cognitive load.

> **tobii-pytracker support:** Fixations and scanpath summaries in this section are produced with the library's `FixationAnalyzer` and `ScanpathsAnalyzer` rather than local event detectors.


In [ ]:
fixations = run_fixations(flat, analysis_dir, FIXATION_PARAMS)
scanpaths = run_scanpaths(fixations, analysis_dir)
fix_summary = summarize_fixations(fixations, len(raw))
scan_summary = summarize_scanpaths(scanpaths, len(raw))

print(f"Detected fixations: {len(fixations)} | Scanpath transitions: {len(scanpaths)}")
display(fixations.head())

## 7. Optional fixation-parameter sensitivity

Fixation detection is parameter-dependent. This optional diagnostic re-runs the built-in dispersion detector at nearby thresholds and reports how strongly event counts change. It is disabled by default to keep routine execution fast.

In [ ]:
sensitivity = pd.DataFrame()
if RUN_FIXATION_SENSITIVITY:
    rows = []
    for threshold in SENSITIVITY_DISPERSION_THRESHOLDS:
        params = dict(FIXATION_PARAMS)
        params["dispersion_threshold"] = float(threshold)
        detected = run_fixations(flat, analysis_dir / "sensitivity", params)
        rows.append({
            "dispersion_threshold": float(threshold),
            "fixation_count": int(len(detected)),
            "mean_fixation_duration": float(detected["duration"].mean()) if not detected.empty else np.nan,
        })
    sensitivity = pd.DataFrame(rows)
    display(sensitivity)
else:
    print("Sensitivity analysis is disabled. Set RUN_FIXATION_SENSITIVITY=True to compare fixation thresholds.")

## 8. Target-phrase AOI analysis

The critical phrase is the experiment's predefined answer-relevant region. The notebook locates the phrase in the stored word geometry and scores fixation centroids against those word boxes. This keeps the analysis tied to the exact spatial representation recorded during presentation.

From the first target fixation, the notebook derives TTFF and pre-target fixation count. Subsequent target entries contribute to dwell and revisit measures, which are useful for describing verification or rereading after acquisition.

> **tobii-pytracker support:** Fixation events come from `FixationAnalyzer` and the word geometry is stored by the experiment output. Matching those events to the experiment-specific `critical_phrase` is notebook logic; the current public `BBoxAttentionAnalyzer` is intended for image bboxes.


In [ ]:

def target_word_boxes(objects_bboxes, phrase: str):
    """Return word AOIs for the first contiguous match of the predefined target phrase."""
    objects = safe_parse(objects_bboxes, dict, {})
    words = objects.get("words", []) if isinstance(objects, dict) else []
    tokens = [normalize_token(w.get("word", "")) for w in words if isinstance(w, dict)]
    target = [normalize_token(token) for token in str(phrase).split()]
    target = [token for token in target if token]
    if not target:
        return []
    width = len(target)
    for start in range(0, len(tokens) - width + 1):
        if tokens[start:start + width] == target:
            return [words[i].get("bbox", {}) for i in range(start, start + width)]
    return []


def target_fixation_metrics(raw: pd.DataFrame, fixations: pd.DataFrame, gaze_flat: pd.DataFrame, phrase_column: str) -> pd.DataFrame:
    """Score fixation centroids against the target phrase without treating missing gaze as target absence."""
    rows = []
    for _, trial in raw.iterrows():
        slide = int(trial["slide_index"])
        boxes = target_word_boxes(trial.get("objects_bboxes"), trial[phrase_column])
        gaze = gaze_flat[gaze_flat["slide_index"] == slide].dropna(subset=["avg_gaze_x", "avg_gaze_y"]) if not gaze_flat.empty else pd.DataFrame()
        has_gaze = not gaze.empty
        f = fixations[fixations["slide_index"] == slide].sort_values("fix_start").copy() if not fixations.empty else pd.DataFrame()

        base = {"slide_index": slide, "target_bbox_found": bool(boxes), "target_bbox_count": int(len(boxes))}
        if not has_gaze:
            rows.append({**base, "target_seen": np.nan, "ttff_target_s": np.nan,
                         "fixations_before_target": np.nan, "pre_target_dwell_s": np.nan,
                         "target_fixation_count": np.nan, "target_dwell_s": np.nan,
                         "target_dwell_share": np.nan, "target_revisits": np.nan,
                         "post_target_fixation_count": np.nan, "post_target_fixation_duration_s": np.nan,
                         "post_target_elapsed_s": np.nan})
            continue
        if f.empty:
            rows.append({**base, "target_seen": False, "ttff_target_s": np.nan,
                         "fixations_before_target": 0, "pre_target_dwell_s": 0.0,
                         "target_fixation_count": 0, "target_dwell_s": 0.0,
                         "target_dwell_share": 0.0, "target_revisits": 0,
                         "post_target_fixation_count": 0, "post_target_fixation_duration_s": 0.0,
                         "post_target_elapsed_s": 0.0})
            continue

        hit = f.apply(lambda r: any(point_in_centered_bbox(r["x_mean"], r["y_mean"], b) for b in boxes), axis=1)
        f = f.assign(target_hit=hit.to_numpy())
        total_dwell = float(f["duration"].sum())
        trial_start = float(gaze["system_time"].min()) if "system_time" in gaze and gaze["system_time"].notna().any() else float(f["fix_start"].min())
        if bool(f["target_hit"].any()):
            first_pos = int(np.flatnonzero(f["target_hit"].to_numpy())[0])
            first = f.iloc[first_pos]
            target_fix = f[f["target_hit"]]
            sequence = f["target_hit"].astype(bool).tolist()
            entries = int(sequence[0]) + sum(sequence[i] and not sequence[i - 1] for i in range(1, len(sequence)))
            revisits = max(entries - 1, 0)
            after = f.iloc[first_pos + 1:]
            target_dwell = float(target_fix["duration"].sum())
            post_elapsed = max(0.0, float(f["fix_end"].max() - first["fix_end"]))
            rows.append({**base, "target_seen": True,
                         "ttff_target_s": max(0.0, float(first["fix_start"] - trial_start)),
                         "fixations_before_target": first_pos,
                         "pre_target_dwell_s": float(f.iloc[:first_pos]["duration"].sum()),
                         "target_fixation_count": int(len(target_fix)),
                         "target_dwell_s": target_dwell,
                         "target_dwell_share": target_dwell / total_dwell if total_dwell else np.nan,
                         "target_revisits": int(revisits),
                         "post_target_fixation_count": int(len(after)),
                         "post_target_fixation_duration_s": float(after["duration"].sum()),
                         "post_target_elapsed_s": post_elapsed})
        else:
            rows.append({**base, "target_seen": False, "ttff_target_s": np.nan,
                         "fixations_before_target": int(len(f)), "pre_target_dwell_s": total_dwell,
                         "target_fixation_count": 0, "target_dwell_s": 0.0,
                         "target_dwell_share": 0.0, "target_revisits": 0,
                         "post_target_fixation_count": 0, "post_target_fixation_duration_s": 0.0,
                         "post_target_elapsed_s": 0.0})
    return pd.DataFrame(rows)


def plot_target_aoi(loader, session, raw, fixations, slide_index: int, phrase_column: str):
    """Overlay the predefined target AOI and detected fixation centroids on the recorded screenshot."""
    from matplotlib.patches import Rectangle
    meta = loader.get_slide_data(session, int(slide_index), flatten=False)
    screenshot = Path(meta["screenshot_path"])
    if not screenshot.exists():
        warnings.warn(f"Screenshot unavailable: {screenshot}")
        return
    trial = raw.loc[raw["slide_index"].eq(int(slide_index))].iloc[0]
    boxes = target_word_boxes(trial.get("objects_bboxes"), trial[phrase_column])
    image = plt.imread(screenshot)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)
    for bbox in boxes:
        try:
            cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
        except Exception:
            continue
        ax.add_patch(Rectangle((width / 2 + cx - w / 2, height / 2 - cy - h / 2), w, h, fill=False, linewidth=2))
    f = fixations[fixations["slide_index"].eq(int(slide_index))] if not fixations.empty else pd.DataFrame()
    if not f.empty:
        ax.scatter(width / 2 + f["x_mean"], height / 2 - f["y_mean"], s=np.maximum(30, f["duration"] * 300), alpha=0.65)
    ax.set_title(f"Target AOI and fixations — slide {slide_index}")
    ax.axis("off")
    plt.show()


In [ ]:

target_metrics = target_fixation_metrics(raw, fixations, flat, "target_phrase")
trial_metrics = (raw[["slide_index","stimulus_id","pair_id","condition","expected_answer","response","correct","uncertain","target_phrase"]]
                 .merge(quality[["slide_index","gaze_samples","usable_gaze","observed_gaze_duration_s"]], on="slide_index")
                 .merge(fix_summary, on="slide_index")
                 .merge(scan_summary, on="slide_index")
                 .merge(target_metrics, on="slide_index"))

eye_cols = ["fixation_count","total_fixation_duration","mean_fixation_duration","scanpath_transition_count","scanpath_distance"]
trial_metrics.loc[~trial_metrics["usable_gaze"], eye_cols] = np.nan
bbox_failures = int((~trial_metrics["target_bbox_found"]).sum())
if bbox_failures:
    warnings.warn(f"Target AOI could not be reconstructed for {bbox_failures} trial(s).")
display(trial_metrics)

if representative is not None:
    plot_target_aoi(loader, SESSION, raw, fixations, representative, "target_phrase")


## 9. Hypothesis-oriented summaries

The tables in this section are organized to mirror the demo documentation rather than to maximize the number of statistics. Read them in the following order:

1. **Target acquisition rate** for H1.
2. **TTFF EARLY vs LATE** for H2.
3. **Pre-target fixation / scanpath measures** for H3.
4. **Accuracy conditional on target acquisition** for H4.

Matched-pair summaries are especially useful here because EARLY and LATE materials are intentionally paired. They remain descriptive in a single-participant demo and should not be interpreted as population-level effects.


In [ ]:

condition_summary = (trial_metrics.groupby("condition", dropna=False)
    .agg(n_trials=("slide_index","size"), accuracy=("correct","mean"), uncertainty_rate=("uncertain","mean"),
         usable_gaze_rate=("usable_gaze","mean"), target_seen_rate=("target_seen","mean"),
         mean_ttff_target_s=("ttff_target_s","mean"), mean_fixations_before_target=("fixations_before_target","mean"),
         mean_pre_target_dwell_s=("pre_target_dwell_s","mean"), mean_target_dwell_s=("target_dwell_s","mean"),
         mean_target_dwell_share=("target_dwell_share","mean"), mean_target_revisits=("target_revisits","mean"),
         mean_fixation_count=("fixation_count","mean"), mean_scanpath_distance=("scanpath_distance","mean"))
    .reset_index())
display(condition_summary)

seen_accuracy = (trial_metrics.dropna(subset=["target_seen"]).groupby("target_seen", dropna=False)
                 .agg(n=("slide_index","size"), accuracy=("correct","mean"))
                 .reset_index())
display(seen_accuracy)

pair_metrics = ["ttff_target_s", "fixations_before_target", "target_dwell_s", "scanpath_distance", "correct"]
pair_wide = trial_metrics.pivot_table(index="pair_id", columns="condition", values=pair_metrics, aggfunc="first")
paired_deltas = pd.DataFrame({"pair_id": pair_wide.index.astype(str)})
for metric in pair_metrics:
    early = pair_wide[(metric, "EARLY")] if (metric, "EARLY") in pair_wide.columns else pd.Series(np.nan, index=pair_wide.index)
    late = pair_wide[(metric, "LATE")] if (metric, "LATE") in pair_wide.columns else pd.Series(np.nan, index=pair_wide.index)
    paired_deltas[f"{metric}_EARLY"] = early.to_numpy()
    paired_deltas[f"{metric}_LATE"] = late.to_numpy()
    if metric != "correct":
        paired_deltas[f"{metric}_LATE_minus_EARLY"] = (late - early).to_numpy()
display(paired_deltas)


## 10. Visual diagnostics

Plots are included to make distribution shape and item-level variation visible. Condition means alone can hide outliers, floor/ceiling effects, and a small number of unusually difficult items.

When reviewing plots, distinguish trials with no target acquisition from trials with missing gaze. The latter are data-quality cases and should not be displayed as meaningful zeroes.


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
trial_metrics.boxplot(column="ttff_target_s", by="condition", ax=axes[0,0])
axes[0,0].set_title("H2: time to first target fixation"); axes[0,0].set_ylabel("seconds")
trial_metrics.boxplot(column="fixations_before_target", by="condition", ax=axes[0,1])
axes[0,1].set_title("H3: fixations before target")
acq = condition_summary.set_index("condition")["target_seen_rate"]
acq.plot(kind="bar", ax=axes[1,0]); axes[1,0].set_ylim(0,1); axes[1,0].set_title("H1: target acquisition rate")
if not seen_accuracy.empty:
    seen_accuracy.set_index("target_seen")["accuracy"].plot(kind="bar", ax=axes[1,1]); axes[1,1].set_ylim(0,1)
axes[1,1].set_title("H4: accuracy by target acquisition")
for ax in axes.flat: ax.set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()

paired_ttff = paired_deltas.dropna(subset=["ttff_target_s_EARLY", "ttff_target_s_LATE"])
if not paired_ttff.empty:
    fig, ax = plt.subplots(figsize=(7,4))
    for _, row in paired_ttff.iterrows():
        ax.plot(["EARLY","LATE"], [row["ttff_target_s_EARLY"], row["ttff_target_s_LATE"]], marker="o", alpha=0.7)
    ax.set_ylabel("TTFF (s)"); ax.set_title("Matched stimulus-pair TTFF")
    plt.show()


## 11. Export reproducible derived tables

Only derived analysis products are written. The original `data.csv` is never modified. Exported tables should be treated as reproducible derivatives of the recorded session and can be regenerated from the provenance information stored earlier in the notebook.

For multi-participant studies, preserve one immutable raw session per participant and combine only the derived trial-level tables in a separate group-analysis workflow.


In [ ]:

trial_metrics.to_csv(analysis_dir / "trial_metrics.csv", index=False)
condition_summary.to_csv(analysis_dir / "condition_summary.csv", index=False)
paired_deltas.to_csv(analysis_dir / "paired_deltas.csv", index=False)
quality.to_csv(analysis_dir / "data_quality_trials.csv", index=False)
quality_overview.to_csv(analysis_dir / "data_quality_summary.csv", index=False)
fixations.to_csv(analysis_dir / "fixations.csv", index=False)
scanpaths.to_csv(analysis_dir / "scanpaths.csv", index=False)
if not sensitivity.empty:
    sensitivity.to_csv(analysis_dir / "fixation_sensitivity.csv", index=False)
provenance.to_csv(analysis_dir / "provenance.csv", index=False)
save_json(analysis_dir / "analysis_parameters.json", {"language": LANGUAGE, "fixation": FIXATION_PARAMS, "sensitivity_thresholds": SENSITIVITY_DISPERSION_THRESHOLDS})
print(f"Derived results written to: {analysis_dir}")


## 12. Scientific interpretation and limitations

**Recommended reading order:** verify design and data quality first, then behavioral accuracy, then target acquisition, and only then event-level gaze metrics. This follows the scientific logic described in the demo documentation: evidence must be interpretable before condition differences are meaningful.

- A lower TTFF in EARLY trials is consistent with faster access to answer-relevant information; it is not a direct measure of cognitive load.
- Missing gaze is unavailable eye-tracking evidence, not failed target acquisition.
- `target_revisits` describes returns after initial acquisition; it can be compatible with verification or ambiguity but does not uniquely identify either process.
- Target dwell can reflect verification, rereading, uncertainty, or visual difficulty and should be interpreted jointly with correctness and response behavior.
- Matched EARLY/LATE deltas are scientifically useful because the items are paired, but a single-session demo cannot establish a population effect.
- Fixation-dependent metrics inherit the assumptions of the fixation detector. Use the sensitivity section before making substantive claims.
- A confirmatory version of this experiment should include repeated participants, preregistered exclusions, counterbalanced materials where appropriate, and participant/item-aware statistical models.

**Documentation link:** `docs/basic_examples/ux_ab_text_demo.md`.
